# Heatmap della scansione (metodo di `movemin.py` dentro la rampa di CH1)

Notebook scritto da zero, indipendente dagli altri, che applica il metodo di `movemin.py` (lasciato com'è) a tutti i punti di un run (cartella con `griglia.csv` e un CSV per punto):

1. **Media mobile di CH2:** la funzione `moving_average` di `movemin.py` copiata identica (`np.convolve` con kernel di `window_length` campioni, reso dispari, `mode="same"`), applicata a tutto CH2. Nonostante il nome del file è una **media mobile**, non un minimo mobile.
2. **Dove misurare:** in `movemin.py` è la fetta fissa `smoothed[5000:60000]`; qui è **la rampa di CH1**, dal primo campione sopra `V_INIZIO_V` fino al massimo di CH1, tolti i `window_length/2` campioni ai bordi del record (dove `mode="same"` usa bordi a zero).
3. **Modulazione** = max - min della media mobile dentro la rampa, e **heatmap** sulle posizioni X, Y del run.

Ordine: cella 1 (scrivi `CARTELLA_RUN`), 2 (calcolo, salva `heatmap_scansione_valori.csv`), 3 (controllo di un punto), 4 (heatmap).

In [ ]:
# Cella 1 - impostazioni e funzioni
import csv
import os
import time

import numpy as np
import matplotlib.pyplot as plt

try:
    import pandas as pd
except ImportError:
    pd = None  # senza pandas la lettura dei CSV grandi e' molto piu' lenta

CARTELLA_RUN = ""          # <-- SCRIVI QUI la cartella del run, es. "2026-10-05/run_002" (relativa al notebook) o un percorso completo
V_INIZIO_V = 0.9           # CH1 sopra questo valore = inizio della rampa (livello del trigger)
TOLLERANZA_FINE_V = 0.3    # la rampa e' finita quando CH1 e' a meno di questo dal suo massimo
WINDOW_LENGTH = 10000      # lunghezza della media mobile in campioni (come window_length di movemin.py)
ASSI_RELATIVI_UM = True    # True: assi in um dal centro della griglia; False: mm
PUNTO_CONTROLLO = 0        # indice del punto mostrato nella cella 3


def leggi_csv(percorso):
    if pd is not None:
        df = pd.read_csv(percorso, dtype=np.float32)
        return df["CH1"].to_numpy(), df["CH2"].to_numpy()
    dati = np.loadtxt(percorso, delimiter=",", skiprows=1, dtype=np.float32)
    return dati[:, 0], dati[:, 1]


def rampa_ch1(ch1):
    """(inizio, fine) della rampa in salita di CH1, oppure None."""
    sopra = np.flatnonzero(ch1 > V_INIZIO_V)
    if sopra.size == 0:
        return None
    i0 = int(sopra[0])
    i1 = i0 + int(np.flatnonzero(ch1[i0:] >= ch1.max() - TOLLERANZA_FINE_V)[0])
    return (i0, i1) if i1 > i0 else None


def moving_average(values, window_length):
    """Media mobile con finestra di lunghezza variabile."""   # identica a movemin.py
    if window_length <= 0:
        raise ValueError('window_length deve essere > 0')

    window_length = int(window_length)
    if window_length % 2 == 0:
        window_length += 1

    kernel = np.ones(window_length) / window_length
    return np.convolve(values, kernel, mode='same')


def modulazione_nella_rampa(ch1, ch2):
    """Restituisce (a, b, max-min della media mobile di CH2 nella rampa di CH1) oppure None.
    a..b e' la zona usata: la rampa senza i window_length/2 campioni ai bordi del record."""
    rampa = rampa_ch1(ch1)
    if rampa is None:
        return None
    smoothed = moving_average(np.asarray(ch2, dtype=float), WINDOW_LENGTH)
    mezza = (WINDOW_LENGTH + (WINDOW_LENGTH % 2 == 0)) // 2
    a = max(rampa[0], mezza)
    b = min(rampa[1], len(ch2) - 1 - mezza)
    if b <= a:
        return None
    curve = smoothed[a:b + 1]
    return a, b, float(np.max(curve) - np.min(curve))


if not CARTELLA_RUN.strip():
    raise ValueError("Scrivi la cartella del run in CARTELLA_RUN, es. '2026-10-05/run_002'")
run_folder = os.path.abspath(CARTELLA_RUN)
with open(os.path.join(run_folder, "griglia.csv"), newline="") as f:
    registro = list(csv.DictReader(f))
print(f"Run: {run_folder} | {len(registro)} punti")

In [ ]:
# Cella 2 - modulazione di ogni punto (dentro la rampa di CH1)
risultati = []   # (indice, x_mm, y_mm, i0, i1, modulazione_V)
t0 = time.time()
for r in registro:
    percorso = os.path.join(run_folder, r["file"])
    if not os.path.isfile(percorso):
        print(f"  punto {r['indice']}: file mancante ({r['file']}), salto")
        continue
    ch1, ch2 = leggi_csv(percorso)
    esito = modulazione_nella_rampa(ch1, ch2)
    if esito is None:
        print(f"  punto {r['indice']}: nessuna rampa in CH1 (mai sopra {V_INIZIO_V} V)")
        risultati.append((int(r["indice"]), float(r["x_mm"]), float(r["y_mm"]), -1, -1, float("nan")))
        continue
    i0, i1, mod = esito
    risultati.append((int(r["indice"]), float(r["x_mm"]), float(r["y_mm"]), i0, i1, mod))
    print(f"  punto {r['indice']}: rampa {i0}..{i1}, modulazione = {mod:.4f} V  [{time.time() - t0:.0f} s]")

valori_path = os.path.join(run_folder, "heatmap_scansione_valori.csv")
with open(valori_path, "w") as f:
    f.write("indice,x_mm,y_mm,i0,i1,modulazione_V\n")
    for riga in risultati:
        f.write(f"{riga[0]},{riga[1]:.4f},{riga[2]:.4f},{riga[3]},{riga[4]},{riga[5]:.6f}\n")
print(f"Salvato: {valori_path}")

In [ ]:
# Cella 3 - controllo di un punto: la zona arancione e' dove si misura (dentro la rampa di CH1)
riga = next((r for r in risultati if r[0] == PUNTO_CONTROLLO), None)
if riga is None:
    raise ValueError(f"Punto {PUNTO_CONTROLLO} non trovato")
indice, x, y, i0, i1, mod = riga
nome_file = next(r["file"] for r in registro if int(r["indice"]) == indice)
ch1, ch2 = leggi_csv(os.path.join(run_folder, nome_file))

passo = max(1, len(ch1) // 5000)
asse = np.arange(len(ch1))[::passo]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
a1.plot(asse, ch1[::passo], color="tab:red", lw=1)
a1.axhline(V_INIZIO_V, color="k", ls="--", lw=0.8)
a1.set_ylabel("CH1 (V)")
a2.plot(asse, ch2[::passo], color="tab:blue", lw=0.6, alpha=0.5, label="CH2 grezzo")
if i0 >= 0:
    smoothed = moving_average(np.asarray(ch2, dtype=float), WINDOW_LENGTH)
    a2.plot(np.arange(len(ch2))[::passo], smoothed[::passo], color="red", lw=1.4, label=f"media mobile (N={WINDOW_LENGTH})")
    for a in (a1, a2):
        a.axvspan(i0, i1, color="orange", alpha=0.25)
a2.set_ylabel("CH2 (V)")
a2.set_xlabel("indice del campione")
a2.legend(loc="best")
a1.set_title(f"Punto {indice}: X={x:.4f} mm, Y={y:.4f} mm | max-min della media mobile nella rampa = {mod:.4f} V")
plt.tight_layout()
plt.show()

In [ ]:
# Cella 4 - heatmap della modulazione
x = np.array([r[1] for r in risultati])
y = np.array([r[2] for r in risultati])
z = np.array([r[5] for r in risultati])

x0, y0 = (x.mean(), y.mean()) if ASSI_RELATIVI_UM else (0.0, 0.0)
k = 1000.0 if ASSI_RELATIVI_UM else 1.0
unita = "um dal centro" if ASSI_RELATIVI_UM else "mm"
xp, yp = (x - x0) * k, (y - y0) * k

xs, ys = np.unique(np.round(xp, 3)), np.unique(np.round(yp, 3))
regolare = (
    len(xs) > 1 and len(ys) > 1 and len(xs) * len(ys) == len(z)
    and np.allclose(np.diff(xs), np.diff(xs).mean(), rtol=1e-2)
    and np.allclose(np.diff(ys), np.diff(ys).mean(), rtol=1e-2)
)

fig, ax = plt.subplots(figsize=(7, 6))
if regolare:
    Z = np.full((len(ys), len(xs)), np.nan)
    for xi, yi, zi in zip(np.round(xp, 3), np.round(yp, 3), z):
        Z[np.searchsorted(ys, yi), np.searchsorted(xs, xi)] = zi
    dx, dy = np.diff(xs).mean(), np.diff(ys).mean()
    im = ax.imshow(Z, origin="lower", aspect="equal", cmap="viridis",
                   extent=[xs[0] - dx / 2, xs[-1] + dx / 2, ys[0] - dy / 2, ys[-1] + dy / 2])
else:
    im = ax.scatter(xp, yp, c=z, s=400, cmap="viridis", marker="s")
    ax.set_aspect("equal")
fig.colorbar(im, ax=ax, label="max - min della media mobile di CH2, nella rampa di CH1 (V)")
ax.set_xlabel(f"X ({unita})")
ax.set_ylabel(f"Y ({unita})")
ax.set_title("Heatmap della scansione")
plt.tight_layout()
percorso_png = os.path.join(run_folder, "heatmap_scansione.png")
fig.savefig(percorso_png, dpi=200)
plt.show()
print(f"Salvata in {percorso_png}")